In [4]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions

In [3]:
df_main = pd.read_excel("data/PO-ini2810_PAHcomplete_20250502.xlsx")
df_main = df_main.drop('Unnamed: 0', axis=1)
df = df_main[['ID', 'Cxsmiles', 'Formula']]
df_main.shape

In [3]:
ctrl = list(df.loc[df_main['Formula'].isna(), 'ID'].unique())
ctrl

['BH4', 'UCR243', 'UCR102', 'DMSO']

In [4]:
df = df.dropna(subset = 'Formula')
df

,ID,Cxsmiles,Formula
2,CSCS00104537142,CC(C(=O)N1CCC(C1)(C(=O)O)C(F)(F)F)N2C=NC=N2,C11H13F3N4O3
3,CSCS00133135693,CC1=NOC(=N1)C2CCN(CC2)C(=O)NC3=CC=C(F)C(F)=C3F,C15H15F3N4O2
4,CSCS00093835758,CN(CC(=O)NC1CCS(=O)(=O)C1)C=2N=CN=C3C=CSC23,C13H16N4O3S2
5,CSCS00133112516,CN1C=CN=C1CN2CCN(CC2)C(=O)NC=3C=CC(=CC3)OC4CCOC4,C20H27N5O3
6,CSCS00133028662,CC1=NN(C=C1CNC2[C@@H]3CCO[C@@H]3C2(C)C)C=4C=CC...,C19H25N3O
...,...,...,...
182683,CSCS00026514704,O=C(C1CC1)N2CCC(CC2)C(=O)N3CCCC=4C=C(F)C=C(F)C43,C19H22F2N2O2
182684,CSCS00106139536,CCC1=NN(CC=2C=CC=CC2)C(CC)=C1CNC=3C=C(Cl)NC(=O)N3,C19H22ClN5O
182685,CSCS00028326356,CN1C=C(C(=O)N2CCN(CC2)C3=CC=CC(Cl)=C3)C=4C=CC=...,C20H20ClN3O
182686,CSCS00103366980,CC(C)N1CCC(CC1)NC(=O)C2=CN=NS2,C11H18N4OS


In [5]:
len(df['ID'].unique()), len(df['Cxsmiles'].unique())

(80000, 80000)

In [6]:
df = df.drop_duplicates(subset='ID')
df

,ID,Cxsmiles,Formula
2,CSCS00104537142,CC(C(=O)N1CCC(C1)(C(=O)O)C(F)(F)F)N2C=NC=N2,C11H13F3N4O3
3,CSCS00133135693,CC1=NOC(=N1)C2CCN(CC2)C(=O)NC3=CC=C(F)C(F)=C3F,C15H15F3N4O2
4,CSCS00093835758,CN(CC(=O)NC1CCS(=O)(=O)C1)C=2N=CN=C3C=CSC23,C13H16N4O3S2
5,CSCS00133112516,CN1C=CN=C1CN2CCN(CC2)C(=O)NC=3C=CC(=CC3)OC4CCOC4,C20H27N5O3
6,CSCS00133028662,CC1=NN(C=C1CNC2[C@@H]3CCO[C@@H]3C2(C)C)C=4C=CC...,C19H25N3O
...,...,...,...
87995,CSCS00133030229,CC(C)(C=1N=NN(CCN2C=C(Cl)C=N2)N1)S(C)(=O)=O,C10H15ClN6O2S
87996,CSCS00027831295,CC1CN(CC(O1)C2CC2)C3=NC=NN3C,C11H18N4O
87997,CSCS00133127202,CC=1N=C(SC1C(=O)O)C(C)NC(=O)C=2C=CC=CC2C3CCC3,C18H20N2O3S
87998,CSCS00025925683,CC(NC(=O)NCC(C)(C)N1CCOCC1)C2=NN=CN2C,C14H26N6O2


In [7]:
df.shape

(80000, 3)

In [8]:
def get_stereoisomers(smiles):
    mol = Chem.MolFromSmiles(smiles)
    opts = StereoEnumerationOptions(onlyUnassigned=True, unique=True)
    isomers = list(EnumerateStereoisomers(mol, options=opts))
    smile_set = []
    for isomer in isomers:
        smile_set.append(Chem.MolToSmiles(isomer, isomericSmiles=True, canonical=True))
    return smile_set

In [9]:
smiles = []
names = []
smiles_isomer = []
names_isomer = []

for _, row in tqdm(df.iterrows()):
    name = row['ID']
    cxsmiles = row['Cxsmiles']

    # Remove counterions if any
    cxsmiles_split = cxsmiles.split('.')
    if len(cxsmiles_split) > 1:
        ## take the last part
        cxsmiles = cxsmiles_split[-1]
    
    # Convert into cononical smiles
    mol = Chem.MolFromSmiles(cxsmiles)
    canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

    # Save the canonical_smiles without isomers
    names.append(name)
    smiles.append(canonical_smiles)
    
    # check if the information of stereocenters are there
    if cxsmiles.find('@') > -1:
        ## If isoform is specified, keep it
        smiles_isomer.append(canonical_smiles)
        names_isomer.append(name)
    else:
        ## Generate all possible isoforms
        stereoisomeric_smiles = get_stereoisomers(canonical_smiles)
        stereoisomeric_names = [f'{name}_{i}' for i in range(len(stereoisomeric_smiles))]
        smiles_isomer.extend(stereoisomeric_smiles)
        names_isomer.extend(stereoisomeric_names)

names_isomer[:10], names_isomer[:10]

80000it [01:10, 1134.41it/s]


(['CSCS00104537142_0',
  'CSCS00104537142_1',
  'CSCS00104537142_2',
  'CSCS00104537142_3',
  'CSCS00133135693_0',
  'CSCS00093835758_0',
  'CSCS00093835758_1',
  'CSCS00133112516_0',
  'CSCS00133112516_1',
  'CSCS00133028662'],
 ['CSCS00104537142_0',
  'CSCS00104537142_1',
  'CSCS00104537142_2',
  'CSCS00104537142_3',
  'CSCS00133135693_0',
  'CSCS00093835758_0',
  'CSCS00093835758_1',
  'CSCS00133112516_0',
  'CSCS00133112516_1',
  'CSCS00133028662'])

In [10]:
len(smiles), len(names), len(smiles_isomer), len(names_isomer)

(80000, 80000, 152202, 152202)

In [24]:
names = ['BH4',
        'Sepiapterin',
        'Phenylalanine',
        'UCR243',
        'UCR102'
       ] + names

smiles = [
    'C[C@H](O)[C@H](O)[C@H]1CNc2nc(N)[nH]c(=O)c2N1',
    'C[C@H](O)C(=O)C1=Nc2c(nc(N)[nH]c2=O)NC1',
    'N[C@@H](Cc1ccccc1)C(=O)O',
    'Clc1ccc2nc(-c3cc[nH]n3)sc2c1',
    'O=c1cc(-c2ccccc2)c2c(O)cc(O)cc2o1'
] + smiles

names_isomer = ['BH4',
                'Sepiapterin',
                'Phenylalanine',
                'UCR243',
                'UCR102'
               ] + names_isomer

smiles_isomer = [
    'C[C@H](O)[C@H](O)[C@H]1CNc2nc(N)[nH]c(=O)c2N1',
    'C[C@H](O)C(=O)C1=Nc2c(nc(N)[nH]c2=O)NC1',
    'N[C@@H](Cc1ccccc1)C(=O)O',
    'Clc1ccc2nc(-c3cc[nH]n3)sc2c1',
    'O=c1cc(-c2ccccc2)c2c(O)cc(O)cc2o1'
    ] + smiles_isomer

In [11]:
len(smiles), len(names), len(smiles_isomer), len(names_isomer)

(80000, 80000, 152202, 152202)

In [12]:
df_new = pd.DataFrame(
    {
        'ID' : names,
        'SMILES' : smiles
    }
)
df_new

,ID,SMILES
0,CSCS00104537142,CC(C(=O)N1CCC(C(=O)O)(C(F)(F)F)C1)n1cncn1
1,CSCS00133135693,Cc1noc(C2CCN(C(=O)Nc3ccc(F)c(F)c3F)CC2)n1
2,CSCS00093835758,CN(CC(=O)NC1CCS(=O)(=O)C1)c1ncnc2ccsc12
3,CSCS00133112516,Cn1ccnc1CN1CCN(C(=O)Nc2ccc(OC3CCOC3)cc2)CC1
4,CSCS00133028662,Cc1nn(-c2ccccc2)cc1CNC1[C@@H]2CCO[C@@H]2C1(C)C
...,...,...
79995,CSCS00133030229,CC(C)(c1nnn(CCn2cc(Cl)cn2)n1)S(C)(=O)=O
79996,CSCS00027831295,CC1CN(c2ncnn2C)CC(C2CC2)O1
79997,CSCS00133127202,Cc1nc(C(C)NC(=O)c2ccccc2C2CCC2)sc1C(=O)O
79998,CSCS00025925683,CC(NC(=O)NCC(C)(C)N1CCOCC1)c1nncn1C


In [13]:
df_isomers = pd.DataFrame(
    {
        'ID' : names_isomer,
        'SMILES' : smiles_isomer
    }
)
df_isomers

,ID,SMILES
0,CSCS00104537142_0,C[C@H](C(=O)N1CC[C@](C(=O)O)(C(F)(F)F)C1)n1cncn1
1,CSCS00104537142_1,C[C@@H](C(=O)N1CC[C@](C(=O)O)(C(F)(F)F)C1)n1cncn1
2,CSCS00104537142_2,C[C@H](C(=O)N1CC[C@@](C(=O)O)(C(F)(F)F)C1)n1cncn1
3,CSCS00104537142_3,C[C@@H](C(=O)N1CC[C@@](C(=O)O)(C(F)(F)F)C1)n1c...
4,CSCS00133135693_0,Cc1noc(C2CCN(C(=O)Nc3ccc(F)c(F)c3F)CC2)n1
...,...,...
152197,CSCS00133127202_0,Cc1nc([C@H](C)NC(=O)c2ccccc2C2CCC2)sc1C(=O)O
152198,CSCS00133127202_1,Cc1nc([C@@H](C)NC(=O)c2ccccc2C2CCC2)sc1C(=O)O
152199,CSCS00025925683_0,C[C@H](NC(=O)NCC(C)(C)N1CCOCC1)c1nncn1C
152200,CSCS00025925683_1,C[C@@H](NC(=O)NCC(C)(C)N1CCOCC1)c1nncn1C


In [14]:
df_new.to_csv("data/ligend_smiles_cononical.csv")

In [27]:
df_isomers.to_excel("data/ligend_smiles_processed.xlsx")
df_isomers.to_csv("data/ligend_smiles_processed.csv")